In [2]:
import platform
import subprocess
import sys

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())

expected_gpu_name = 'T4'
expected_min_gpus = 2

try:
    import torch
except Exception as e:
    torch = None
    print('Torch import failed:', repr(e))

if torch is not None:
    print('Torch version:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())

    if torch.cuda.is_available():
        device_count = torch.cuda.device_count()
        print('CUDA device count:', device_count)

        device_names = []
        for i in range(device_count):
            name = torch.cuda.get_device_name(i)
            device_names.append(name)
            print(f'GPU {i}: {name}')

        gpu_name_ok = all(expected_gpu_name.upper() in n.upper() for n in device_names)
        gpu_count_ok = device_count >= expected_min_gpus

        if gpu_name_ok and gpu_count_ok:
            print(f'{expected_gpu_name} x{expected_min_gpus} check: PASS')
        else:
            print(f'{expected_gpu_name} x{expected_min_gpus} check: FAIL')
            print('  - gpu_name_ok =', gpu_name_ok)
            print('  - gpu_count_ok =', gpu_count_ok)

        try:
            x = torch.tensor([1.0], device='cuda')
            y = (x * 2).item()
            print('CUDA tensor op: PASS', y)
        except Exception as e:
            print('CUDA tensor op: FAIL', repr(e))
    else:
        print('No CUDA GPU detected by torch.')

try:
    smi = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
        capture_output=True,
        text=True,
        timeout=10,
    )
    if smi.returncode == 0:
        print('nvidia-smi:', smi.stdout.strip())
    else:
        print('nvidia-smi failed:', smi.stderr.strip() or f'returncode={smi.returncode}')
except Exception as e:
    print('nvidia-smi not available:', repr(e))

Python: 3.12.12
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
Torch version: 2.10.0+cu128
CUDA available: True
CUDA device count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
T4 x2 check: PASS
CUDA tensor op: PASS 2.0
nvidia-smi: Tesla T4, 15360 MiB, 580.105.08
Tesla T4, 15360 MiB, 580.105.08


In [7]:
from pathlib import Path

required_files = ['train.csv', 'test.csv']
optional_files = ['sample_submission.csv']


def has_required_files(directory: Path) -> bool:
    return directory.is_dir() and all((directory / f).exists() for f in required_files)


candidate_dirs = []
kaggle_input = Path('/kaggle/input')

# Prefer Kaggle-mounted datasets first.
if kaggle_input.exists() and kaggle_input.is_dir():
    print('Found /kaggle/input')

    top_level_dirs = [d for d in kaggle_input.iterdir() if d.is_dir()]
    print('Top-level dirs under /kaggle/input:', len(top_level_dirs))
    for d in top_level_dirs[:20]:
        print('  -', d)

    for ds_dir in top_level_dirs:
        candidate_dirs.append(ds_dir)
        candidate_dirs.append(ds_dir / 'public')
        candidate_dirs.append(ds_dir / 'dataset')
        candidate_dirs.append(ds_dir / 'dataset' / 'public')

    # Recursive fallback: detect any folder containing both train.csv and test.csv.
    for train_file in kaggle_input.rglob('train.csv'):
        parent = train_file.parent
        if (parent / 'test.csv').exists():
            candidate_dirs.append(parent)
else:
    print('/kaggle/input not found in this runtime.')

# Local/workspace fallbacks.
candidate_dirs.extend([
    Path('./dataset/public'),
    Path('../dataset/public'),
    Path('./dataset'),
    Path('../dataset'),
])

found_dirs = []
seen = set()
for directory in candidate_dirs:
    try:
        key = str(directory.resolve()) if directory.exists() else str(directory)
    except Exception:
        key = str(directory)

    if key in seen:
        continue
    seen.add(key)

    if has_required_files(directory):
        found_dirs.append(directory.resolve())

print('Working directory:', Path.cwd())
print('Valid dataset directories found:', len(found_dirs))

if found_dirs:
    for idx, directory in enumerate(found_dirs, 1):
        print(f'{idx}. {directory}')

    DATA_DIR = found_dirs[0]
    print('Using DATA_DIR =', DATA_DIR)

    for fname in required_files + optional_files:
        fpath = DATA_DIR / fname
        if fpath.exists():
            print(f'  - {fname}: FOUND')
        else:
            print(f'  - {fname}: MISSING (optional)')
else:
    DATA_DIR = None
    print('No dataset directory found.')
    print('Attach dataset in Kaggle (Add Data), then re-run this cell.')

Found /kaggle/input
Top-level dirs under /kaggle/input: 1
  - /kaggle/input/datasets
Working directory: /kaggle/working
Valid dataset directories found: 1
1. /kaggle/input/datasets/soufianelasfar/onnx-dataset
Using DATA_DIR = /kaggle/input/datasets/soufianelasfar/onnx-dataset
  - train.csv: FOUND
  - test.csv: FOUND
  - sample_submission.csv: FOUND
